# Sobrino2008 — Emissividade

Estima emissividade por mistura solo/vegetação, na variante simplificada ou completa com cavidade. Os parâmetros devem ser compatíveis com a banda e a superfície.

## Precisa de

**Bandas ópticas — obrigatórias na forma de uso deste exemplo:**

- **Vermelho:** imagem calibrada em reflectância de superfície, **adimensional**.
- **Infravermelho próximo (NIR):** imagem calibrada na **mesma grandeza e unidade do vermelho**.

A função calcula o NDVI internamente. Como alternativa, fornecer somente `ndvi_image` já preparado, **adimensional, em [-1,1]**, no lugar das duas bandas; não combinar as entradas.

**Banda de destino:**

- **`band`:** número da banda, **sem unidade**; identifica a banda dos parâmetros fornecidos; não atribui endmembers automaticamente.

**Parâmetros obrigatórios nas duas variantes:**

- **`soil_emissivity`:** emissividade do solo, **adimensional, em (0,1]**; número único ou imagem alinhada.
- **`vegetation_emissivity`:** emissividade da vegetação, **adimensional, em (0,1]**; número único ou imagem alinhada.

Fornecer valores verificados para a banda. Os endmembers demonstrativos do notebook não são padrões universais de Sobrino.

**Somente se escolher `variant="full"`:**

- **`red_surface` — obrigatório:** imagem vermelha em **reflectância de superfície, adimensional, em [0,1]**.
- **`soil_red_coefficients` — obrigatório:** par **(a,b), adimensional**, da relação emissividade do solo = a + b × reflectância vermelha; verificar os coeficientes da banda.
- **`cavity_factor` — opcional:** fator geométrico de cavidade, **adimensional**; padrão **0,55**.
- **`vegetation_cavity` — opcional:** contribuição de cavidade na vegetação, **adimensional**; padrão **0,005**.

A variante `"simplified"` não utiliza esses quatro parâmetros. Os extremos NDVI, adimensionais, são configuráveis e detalhados na tabela.

**Entrada opcional de qualidade:**

- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar, sujeito às validações do método. Neste exemplo, sua origem é descrita na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

- **Vermelho superficial:** GeoTIFF com valores físicos de **reflectância de superfície, adimensionais**.
- **NIR superficial:** GeoTIFF com valores físicos de **reflectância de superfície, adimensionais**.

O leitor confere a grade e usa os NoData dos arquivos. Não monta automaticamente uma máscara de nuvens: complemente a máscara com a qualidade e as classes inadequadas para o método. Não é necessário fornecer MTL nem bandas térmicas para calcular somente a emissividade a partir dessas imagens prontas.

## Como as imagens são preparadas

Este notebook recebe vermelho e NIR **já em reflectância de superfície**, adimensional. Ele lê os GeoTIFFs e seus NoData, mas não converte DN em reflectância e não aplica correção atmosférica.

Ao usar um produto de reflectância de superfície com valores inteiros escalados, aplique antes o fator de escala e o offset documentados pelo fornecedor. Não usar os inteiros armazenados como se já fossem reflectância física.

Reflectância TOA (topo da atmosfera) e reflectância de superfície são grandezas diferentes. A conversão de DN para TOA, sozinha, não produz a reflectância superficial necessária neste exemplo.

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = emissivity(
    # Método
    method='sobrino-2008',

    # Banda de destino
    band=10,

    # Parametrização da emissividade
    variant='simplified',
    soil_emissivity=emissividade_solo,
    vegetation_emissivity=emissividade_vegetacao,
    ndvi_soil=0.2,
    ndvi_vegetation=0.5,

    # Entradas para o NDVI
    red=vermelho,
    nir=nir,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "sobrino-2008" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None` | Número da banda | Identifica a banda dos endmembers fornecidos; inteiro positivo, sem atribuição automática de constantes. |
| `variant` | Opcional | `None → "simplified"` | Seletor | "simplified": mistura solo/vegetação; "full": relação do solo e cavidade, com parâmetros adicionais específicos da banda. |
| `soil_emissivity` | Obrigatório em Sobrino2008 | `None` | Adimensional, (0,1] | Emissividade de solo específica da banda, escalar ou raster alinhado. Registrar origem e compatibilidade com sensor/superfície; não há valor universal. |
| `vegetation_emissivity` | Obrigatório em Sobrino2008 | `None` | Adimensional, (0,1] | Emissividade de vegetação específica da banda, escalar ou raster alinhado. Registrar origem e compatibilidade com sensor/superfície; não há valor universal. |
| `ndvi_soil` | Opcional | `None → 0.2` | Adimensional, [-1,1] | Extremo NDVI de solo. ndvi_soil deve ser menor que ndvi_vegetation. Justificar os valores para a cena. |
| `ndvi_vegetation` | Opcional | `None → 0.5` | Adimensional, [-1,1] | Extremo NDVI de vegetação. ndvi_soil deve ser menor que ndvi_vegetation. Justificar os valores para a cena. |
| `red` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | Reflectância superficial (adimensional) | Banda vermelha calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `nir` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | Reflectância superficial (adimensional) | Banda NIR calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — NDVI preparado

Usar NDVI já calculado e alinhado no lugar do par vermelho/NIR. Não há necessidade de chamar uma função pública de NDVI.

```python
resultado = emissivity(
    # Método
    method='sobrino-2008',

    # Banda de destino
    band=10,

    # Parametrização da emissividade
    variant='simplified',
    soil_emissivity=emissividade_solo,
    vegetation_emissivity=emissividade_vegetacao,
    ndvi_soil=0.2,
    ndvi_vegetation=0.5,

    # Entradas para o NDVI
    ndvi_image=ndvi_preparado,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "sobrino-2008" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None` | Número da banda | Identifica a banda dos endmembers fornecidos; inteiro positivo, sem atribuição automática de constantes. |
| `variant` | Opcional | `None → "simplified"` | Seletor | "simplified": mistura solo/vegetação; "full": relação do solo e cavidade, com parâmetros adicionais específicos da banda. |
| `soil_emissivity` | Obrigatório em Sobrino2008 | `None` | Adimensional, (0,1] | Emissividade de solo específica da banda, escalar ou raster alinhado. Registrar origem e compatibilidade com sensor/superfície; não há valor universal. |
| `vegetation_emissivity` | Obrigatório em Sobrino2008 | `None` | Adimensional, (0,1] | Emissividade de vegetação específica da banda, escalar ou raster alinhado. Registrar origem e compatibilidade com sensor/superfície; não há valor universal. |
| `ndvi_soil` | Opcional | `None → 0.2` | Adimensional, [-1,1] | Extremo NDVI de solo. ndvi_soil deve ser menor que ndvi_vegetation. Justificar os valores para a cena. |
| `ndvi_vegetation` | Opcional | `None → 0.5` | Adimensional, [-1,1] | Extremo NDVI de vegetação. ndvi_soil deve ser menor que ndvi_vegetation. Justificar os valores para a cena. |
| `ndvi_image` | Condicional: alternativa ao par vermelho/NIR | `None` | Adimensional, [-1,1] | NDVI já preparado e alinhado; omitir as duas bandas ópticas. A convenção radiométrica deve respeitar a fonte do método. Não combinar NDVI preparado e vermelho/NIR. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — Sobrino2008 completo

Fornecer coeficientes_solo=(a,b) verificados para a banda. Os endmembers deste exemplo são demonstrativos; a variante full não cria coeficientes de solo automaticamente.

```python
resultado = emissivity(
    # Método
    method='sobrino-2008',

    # Banda de destino
    band=10,

    # Parametrização da emissividade
    variant='full',
    soil_emissivity=emissividade_solo,
    vegetation_emissivity=emissividade_vegetacao,
    ndvi_soil=0.2,
    ndvi_vegetation=0.5,

    # Entradas para o NDVI
    red=vermelho,
    nir=nir,

    # Relação de solo e cavidade — somente full
    red_surface=vermelho,
    soil_red_coefficients=coeficientes_solo,
    cavity_factor=0.55,
    vegetation_cavity=0.005,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "sobrino-2008" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None` | Número da banda | Identifica a banda dos endmembers fornecidos; inteiro positivo, sem atribuição automática de constantes. |
| `variant` | Opcional | `None → "simplified"` | Seletor | "simplified": mistura solo/vegetação; "full": relação do solo e cavidade, com parâmetros adicionais específicos da banda. |
| `soil_emissivity` | Obrigatório em Sobrino2008 | `None` | Adimensional, (0,1] | Emissividade de solo específica da banda, escalar ou raster alinhado. Registrar origem e compatibilidade com sensor/superfície; não há valor universal. |
| `vegetation_emissivity` | Obrigatório em Sobrino2008 | `None` | Adimensional, (0,1] | Emissividade de vegetação específica da banda, escalar ou raster alinhado. Registrar origem e compatibilidade com sensor/superfície; não há valor universal. |
| `ndvi_soil` | Opcional | `None → 0.2` | Adimensional, [-1,1] | Extremo NDVI de solo. ndvi_soil deve ser menor que ndvi_vegetation. Justificar os valores para a cena. |
| `ndvi_vegetation` | Opcional | `None → 0.5` | Adimensional, [-1,1] | Extremo NDVI de vegetação. ndvi_soil deve ser menor que ndvi_vegetation. Justificar os valores para a cena. |
| `red` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | Reflectância superficial (adimensional) | Banda vermelha calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `nir` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | Reflectância superficial (adimensional) | Banda NIR calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `red_surface` | Obrigatório somente em full | `None` | Reflectância superficial, [0,1] | Imagem vermelha de superfície alinhada; não substituir por DN, radiância ou TOA. Omitir em simplified. |
| `soil_red_coefficients` | Obrigatório somente em full | `None` | (a,b) adimensionais | Par finito da relação de solo emissividade = a + b × red_surface, verificado para a banda. Omitir em simplified. |
| `cavity_factor` | Opcional somente em full | `None → 0.55` | Adimensional, [0,1] | Fator geométrico de cavidade. Omitir em simplified; o padrão é uma hipótese do modelo. |
| `vegetation_cavity` | Opcional somente em full | `None → 0.005` | Adimensional, [0,1) | Contribuição adicional de cavidade na vegetação. Omitir em simplified. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import emissivity
import rasterio

vermelho_path = Path("CAMINHO_DO_VERMELHO_REFLECTANCIA.tif")
nir_path = Path("CAMINHO_DO_NIR_REFLECTANCIA.tif")
saida = Path("resultados/sobrino_2008")
saida.mkdir(parents=True, exist_ok=True)
banda = 10
variante = "simplified"
# Valores demonstrativos da banda10 de Rongali; nao sao defaults universais Sobrino.
emissividade_solo = 0.971
emissividade_vegetacao = 0.987
coeficientes_solo = None  # full requer (a,b) verificados para a banda
ndvi_solo, ndvi_vegetacao = 0.2, 0.5
fator_cavidade, cavidade_vegetacao = 0.55, 0.005

## Ler as imagens

As duas bandas precisam estar na mesma grade e ja em reflectancia. O exemplo nao transforma DN em reflectancia de superficie nem calcula correcao atmosferica. `rasterio.open` le os arquivos e fornece o perfil de georreferenciamento.

### Máscara usada neste exemplo

Os arquivos de reflectância são lidos com seus NoData. A máscara booleana reúne pixels não finitos; True exclui. Complemente-a com nuvens, sombras, água/neve e classes incompatíveis com o modelo usando a informação de qualidade dos seus produtos. O leitor destes dois rasters não cria QA de nuvens automaticamente.


In [ ]:
with rasterio.open(vermelho_path) as src:
    vermelho = src.read(1, masked=True).astype(float).filled(np.nan)
    perfil = src.profile.copy()
    grade = (src.shape, src.crs, src.transform)
with rasterio.open(nir_path) as src:
    if (src.shape, src.crs, src.transform) != grade:
        raise ValueError("As bandas vermelho/NIR precisam estar alinhadas.")
    nir = src.read(1, masked=True).astype(float).filled(np.nan)
mascara = ~np.isfinite(vermelho) | ~np.isfinite(nir)

## Aplicar o metodo

A combinacao dos endmembers demonstrativos com Sobrino e uma adaptacao explicita. Para B11 ou outro sensor/superficie, forneca os parametros correspondentes.

In [ ]:
opcoes_full = {}
if variante == 'full':
    if coeficientes_solo is None:
        raise ValueError('Informe coeficientes_solo=(a,b) verificados para usar full.')
    opcoes_full = dict(red_surface=vermelho, soil_red_coefficients=coeficientes_solo, cavity_factor=fator_cavidade, vegetation_cavity=cavidade_vegetacao)
imagem = emissivity(
    # Método
    method='sobrino-2008',

    # Banda de destino
    band=banda,

    # Parametrização da emissividade
    variant=variante,
    soil_emissivity=emissividade_solo,
    vegetation_emissivity=emissividade_vegetacao,
    ndvi_soil=ndvi_solo,
    ndvi_vegetation=ndvi_vegetacao,

    # Entradas para o NDVI
    red=vermelho,
    nir=nir,

    # Relação de solo e cavidade — somente full
    red_surface=opcoes_full.get('red_surface'),
    soil_red_coefficients=opcoes_full.get('soil_red_coefficients'),
    cavity_factor=opcoes_full.get('cavity_factor'),
    vegetation_cavity=opcoes_full.get('vegetation_cavity'),

    # Exclusão de pixels
    mask=mascara,
)


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
perfil.update(dtype="float32", count=1, nodata=np.nan)
with rasterio.open(saida / "emissividade.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units="dimensionless")
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

[Artigo ou documentacao original](https://doi.org/10.1109/TGRS.2007.904834).

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).

Para a variante full, altere `variante`, forneca `coeficientes_solo` e utilize vermelho de superficie. Mantenha parametros de cavidade apenas nesse modo.